# La croissance nourrit les bénéfices · *Growth feeds profits*

Notebook compagnon du chapitre **4. Les grandes variables macro qui font bouger vos placements : un panorama** — [lire l'article](https://nmlab.io/ressources/les-grandes-variables-macro-qui-font-bouger-vos-placements).
Companion notebook to chapter **4. The Big Macro Variables That Move Your Investments: A Panorama** — [read the article](https://nmlab.io/en/ressources/the-big-macro-variables-that-move-your-investments).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_data() -> tuple[Series, Series, Series]:
    """PIB nominal (GDP) et bénéfices des entreprises après impôts (CP), trimestriels,
    plus l'indicateur de récession NBER (USREC) — en direct de FRED, depuis 1990.
    Nominal GDP, after-tax corporate profits and the NBER recession flag, live from FRED."""
    gdp = nm.load_fred("GDP", start="1990")
    profits = nm.load_fred("CP", start="1990")
    recessions = nm.load_fred("USREC", start="1990")
    return gdp, profits, recessions

gdp, profits, recessions = load_data()


import matplotlib.dates as mdates
from pandas import Series
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="La croissance nourrit les bénéfices",
        sub="États-Unis, 1990-2025 : PIB et bénéfices des entreprises, base 100 en 1990",
        ylab="indice, base 100 en 1990",
        legend=["PIB (nominal)", "bénéfices des entreprises"],
        note="Les bénéfices amplifient les cycles de l'activité, mais en épousent la pente. Bandes : récessions\n"
             "NBER. Source : BEA et NBER via FRED (GDP, CP, USREC)."),
    "en": dict(
        title="Growth feeds profits",
        sub="United States, 1990-2025: GDP and corporate profits, indexed to 100 in 1990",
        ylab="index, 100 = 1990",
        legend=["GDP (nominal)", "corporate profits"],
        note="Profits amplify the cycles of activity but follow its slope. Bands: NBER recessions.\n"
             "Source: BEA and NBER via FRED (GDP, CP, USREC)."),
}

def build_figure(gdp: Series, profits: Series, recessions: Series, lang: str) -> Figure:
    """Deux séries en base 100 (PIB nominal et bénéfices), ombrées des récessions du NBER."""
    text = LABELS[lang]
    g = gdp / gdp.iloc[0] * 100
    p = profits / profits.iloc[0] * 100
    fig = nm.figure(height_px=1102)
    ax = nm.axes(fig, left=0.088, right=0.9)

    runs = recessions.ne(recessions.shift()).cumsum()
    for _, run in recessions.groupby(runs):
        if run.iloc[0] == 1:
            ax.axvspan(run.index[0], run.index[-1], color=nm.COLORS["edge"], alpha=0.75, linewidth=0)

    ax.plot(g.index, g, color=nm.COLORS["blue"], linewidth=2.9, label=text["legend"][0])
    ax.plot(p.index, p, color=nm.COLORS["rose"], linewidth=2.9, label=text["legend"][1])
    ax.set_ylim(60, 1520)
    ax.set_yticks(range(200, 1401, 200))
    ax.yaxis.set_major_formatter(nm.thousands(lang))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    for series, color in ((p, nm.COLORS["rose"]), (g, nm.COLORS["blue"])):
        mult = series.iloc[-1] / 100
        label = (f"× {mult:.1f}".replace(".", ",") if lang == "fr" else f"× {mult:.1f}")
        ax.annotate(label, xy=(series.index[-1], series.iloc[-1]), xytext=(14, 0),
                    textcoords="offset points", ha="left", va="center",
                    fontsize=28, fontweight="bold", color=color)

    leg = ax.legend(loc="upper left", frameon=True, fontsize=22, labelcolor=nm.COLORS["text"],
                    handlelength=1.6, borderpad=0.9)
    leg.get_frame().set_facecolor(nm.COLORS["card"])
    leg.get_frame().set_edgecolor(nm.COLORS["edge"])
    leg.get_frame().set_linewidth(1.4)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(gdp, profits, recessions, LANG)